<a href="https://colab.research.google.com/github/LGBlack098/mi-repositorio-ed1/blob/main/Ejercicio_3_Simulaci%C3%B3n_de_Datos_Estructuras_de_Datos_INF_220_UAGRM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estructuras de Datos - INF 220 **[UAGRM]**
## Unidad I: Modelos de Representación de Datos
### Ejercicio 3: Simulación y Comparación Empírica (Estático vs Dinámico)

**Objetivos:**
- Evaluar cuantitativamente el rendimiento de las estructuras estáticas y dinámicas implementadas en el Ejercicio 2.
- Generar un conjunto sintético de 50 registros de `Estudiante` y medir:
  1. **Tiempo de inserción** ($O(1)$ al final).
  2. **Tiempo de búsqueda secuencial** en el peor caso (elemento final).
  3. **Consumo aproximado de memoria** (overhead de punteros y referencias).


### 1. Definición de Entidades y Estructuras Base


In [ ]:
from dataclasses import dataclass
import random
import sys
import time

# --- Estructura Estática ---
class ArrayEstatico:
    def __init__(self, capacidad):
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser positiva")
        self._capacidad = capacidad
        self._datos = [None] * capacidad
        self._n = 0

    def insertar_final(self, item):
        if self._n >= self._capacidad:
            raise OverflowError("Desbordamiento: capacidad máxima alcanzada")
        self._datos[self._n] = item
        self._n += 1

    def __iter__(self):
        for i in range(self._n):
            yield self._datos[i]

# --- Estructura Dinámica ---
class _Nodo:
    __slots__ = ("dato", "siguiente")
    def __init__(self, dato, siguiente=None):
        self.dato = dato
        self.siguiente = siguiente

class ListaDinamica:
    def __init__(self):
        self._cabeza = None
        self._cola = None
        self._n = 0

    def insertar_final(self, item):
        nuevo = _Nodo(item)
        if self._cola is None:
            self._cabeza = self._cola = nuevo
        else:
            self._cola.siguiente = nuevo
            self._cola = nuevo
        self._n += 1

    def __iter__(self):
        actual = self._cabeza
        while actual is not None:
            yield actual.dato
            actual = actual.siguiente

# --- Entidad de Dominio ---
@dataclass
class Estudiante:
    nombre: str
    nota: float
    grupo: str


### 2. Generador de Datos y Algoritmos de Búsqueda


In [ ]:
def generar_estudiantes(n=50, semilla=42):
    random.seed(semilla)
    nombres = ["Ana", "Luis", "María", "Carlos", "Elena", "Pedro", "Sofía", "Diego",
               "Valeria", "Jorge", "Lucía", "Andrés", "Camila", "Fernando", "Gabriela"]
    return [
        Estudiante(
            nombre=random.choice(nombres),
            nota=round(random.uniform(1.0, 7.0), 1),
            grupo=random.choice(["A", "B", "C"]),
        )
        for _ in range(n)
    ]

def buscar_array(array: ArrayEstatico, nombre: str):
    """Búsqueda lineal en ArrayEstatico. Devuelve el primer Estudiante hallado."""
    for est in array:
        if est.nombre == nombre:
            return est
    return None

def buscar_lista(lista: ListaDinamica, nombre: str):
    """Búsqueda lineal en ListaDinamica. Devuelve el primer Estudiante hallado."""
    for est in lista:
        if est.nombre == nombre:
            return est
    return None


### 3. Ejecución de la Simulación y Benchmarking


In [ ]:
def simular_carga_y_busqueda():
    estudiantes = generar_estudiantes(50)
    nombre_objetivo = estudiantes[-1].nombre  # peor caso: al final de ambas estructuras

    # --- Carga ---
    array = ArrayEstatico(100)
    t0 = time.perf_counter()
    for est in estudiantes:
        array.insertar_final(est)
    tiempo_array = time.perf_counter() - t0

    lista = ListaDinamica()
    t0 = time.perf_counter()
    for est in estudiantes:
        lista.insertar_final(est)
    tiempo_lista = time.perf_counter() - t0

    # --- Búsqueda (promedio de varias repeticiones para estabilizar la medición) ---
    reps = 2000
    t0 = time.perf_counter()
    for _ in range(reps):
        buscar_array(array, nombre_objetivo)
    tiempo_busq_array = (time.perf_counter() - t0) / reps

    t0 = time.perf_counter()
    for _ in range(reps):
        buscar_lista(lista, nombre_objetivo)
    tiempo_busq_lista = (time.perf_counter() - t0) / reps

    # --- Memoria aproximada (solo contenedores, sin contar los objetos Estudiante) ---
    mem_array = sys.getsizeof(array._datos)
    mem_lista = sys.getsizeof(lista) + lista._n * 48  # aprox. 48 bytes/nodo con __slots__

    print("=" * 60)
    print("RESULTADOS DE LA SIMULACIÓN (50 Registros)")
    print("=" * 60)
    print(f"Tiempo de inserción ArrayEstatico:  {tiempo_array:.6f} s")
    print(f"Tiempo de inserción ListaDinamica:  {tiempo_lista:.6f} s")
    print(f"Tiempo de búsqueda ArrayEstatico:    {tiempo_busq_array * 1000:.6f} ms (promedio, n={reps})")
    print(f"Tiempo de búsqueda ListaDinamica:    {tiempo_busq_lista * 1000:.6f} ms (promedio, n={reps})")
    print(f"Memoria aproximada ArrayEstatico (capacidad 100): {mem_array} bytes")
    print(f"Memoria aproximada ListaDinamica (50 nodos):      {mem_lista} bytes")
    print("=" * 60)

    return estudiantes

estudiantes_generados = simular_carga_y_busqueda()


### 4. Análisis e Interpretación de Resultados

- **Tiempo de inserción:** Ambas operaciones son $O(1)$ para inserción al final. El arreglo suele ser ligeramente más veloz gracias a la localidad espacial de memoria en caché y la ausencia de asignación individual de objetos nodo en el Heap.
- **Facilidad y velocidad de búsqueda:** En ambas estructuras desordenadas se requiere un recorrido lineal $O(n)$. No obstante, el arreglo almacena punteros continuos que aprovechan mejor la línea de caché de la CPU. Además, si estuviera ordenado, el arreglo admitiría búsqueda binaria $O(\log n)$ por acceso directo $O(1)$.
- **Consumo de memoria:** La lista enlazada dinámica presenta una mayor sobrecarga (*memory overhead*) debido a que cada nodo individual requiere memoria para el descriptor de objeto y los punteros (`dato` y `siguiente`), mientras que el arreglo estático solo reserva una tabla contigua de referencias.
